# Non-Overlapping 75-Day Windows

This notebook imports the wildfire dataset, validates complete 75-day windows per coordinate, removes overlapping windows, and assigns final `window_id` values.

Important: the notebook does not sort by `latitude`, `longitude`, and `datetime` before windowing. It preserves the source row order inside each coordinate so overlapping blocks do not get interleaved.

In [5]:
import pandas as pd
import numpy as np

In [6]:
SEQ_LEN = 75

FEATURES = [
    "pr", "rmax", "rmin", "sph", "srad", "tmmn", "tmmx",
    "vs", "bi", "fm100", "fm1000", "erc", "etr", "pet", "vpd"
]

df = pd.read_csv("Wildfire_Dataset.csv", parse_dates=["datetime"])

df["Wildfire"] = (
    df["Wildfire"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"no": 0, "yes": 1})
)

if df["Wildfire"].isna().any():
    raise ValueError("Unexpected values found in Wildfire column.")

# Preserve source order so pre-existing 75-row blocks are not interleaved.
df = df.reset_index(drop=False).rename(columns={"index": "source_row"})

print(f"Rows loaded: {len(df):,}")
print(f"Unique coordinates: {df[[ 'latitude', 'longitude' ]].drop_duplicates().shape[0]:,}")

Rows loaded: 9,509,925
Unique coordinates: 37,098


In [7]:
# Step 2: build candidate 75-day windows in original per-coordinate row order
df["row_in_coord"] = df.groupby(["latitude", "longitude"], sort=False).cumcount()
df["window_in_coord"] = df["row_in_coord"] // SEQ_LEN

df["candidate_window_id"] = (
    df["latitude"].astype(str) + "_" +
    df["longitude"].astype(str) + "_" +
    df["window_in_coord"].astype(str)
)

window_sizes = df.groupby("candidate_window_id").size()
complete_window_ids = window_sizes[window_sizes == SEQ_LEN].index

df_75 = df[df["candidate_window_id"].isin(complete_window_ids)].copy()
df_75 = df_75.sort_values(["candidate_window_id", "source_row"]).reset_index(drop=True)

print(f"Complete 75-day windows: {len(complete_window_ids):,}")
print(f"Rows retained in complete windows: {len(df_75):,}")
display(df_75.head())

Complete 75-day windows: 126,799
Rows retained in complete windows: 9,509,925


,source_row,latitude,longitude,datetime,Wildfire,pr,rmax,rmin,sph,srad,...,bi,fm100,fm1000,erc,etr,pet,vpd,row_in_coord,window_in_coord,candidate_window_id
0,297300,25.26027,-80.80099,2016-04-10,0,0.0,91.2,44.7,0.01111,306.6,...,37.0,16.4,18.6,27.0,8.0,5.9,0.71,0,0,25.2602699992712_-80.80099_0
1,297301,25.26027,-80.80099,2016-04-11,0,0.0,100.0,51.0,0.01295,242.3,...,32.0,16.7,18.5,25.0,5.9,4.6,0.75,1,0,25.2602699992712_-80.80099_0
2,297302,25.26027,-80.80099,2016-04-12,0,0.0,100.0,45.2,0.01282,299.7,...,27.0,16.7,18.4,27.0,6.6,5.3,0.79,2,0,25.2602699992712_-80.80099_0
3,297303,25.26027,-80.80099,2016-04-13,0,0.0,100.0,47.0,0.01364,326.5,...,26.0,16.8,18.3,27.0,6.5,5.5,0.74,3,0,25.2602699992712_-80.80099_0
4,297304,25.26027,-80.80099,2016-04-14,0,1.0,100.0,49.7,0.01437,292.7,...,27.0,16.9,18.2,27.0,6.4,5.2,0.78,4,0,25.2602699992712_-80.80099_0


In [8]:
# Step 3: remove any window that overlaps another window on the same coordinate-date row
overlap_counts = (
    df_75.groupby(["latitude", "longitude", "datetime"])
    .size()
    .rename("n_rows")
    .reset_index()
)

overlap_pairs = overlap_counts[overlap_counts["n_rows"] > 1].copy()

overlap_window_ids = set()
if not overlap_pairs.empty:
    overlap_window_ids = set(
        df_75.merge(
            overlap_pairs[["latitude", "longitude", "datetime"]],
            on=["latitude", "longitude", "datetime"],
            how="inner",
        )["candidate_window_id"].unique()
    )

df_non_overlap = df_75[~df_75["candidate_window_id"].isin(overlap_window_ids)].copy()

print(f"Duplicate coordinate-date pairs among 75-day windows: {len(overlap_pairs):,}")
print(f"Candidate windows removed for overlap: {len(overlap_window_ids):,}")
print(f"Non-overlapping 75-day windows kept: {df_non_overlap['candidate_window_id'].nunique():,}")
print(f"Rows kept after overlap removal: {len(df_non_overlap):,}")

if not overlap_pairs.empty:
    display(overlap_pairs.head())

Duplicate coordinate-date pairs among 75-day windows: 17,063
Candidate windows removed for overlap: 804
Non-overlapping 75-day windows kept: 125,995
Rows kept after overlap removal: 9,449,625


,latitude,longitude,datetime,n_rows
100457,28.425833,-81.395,2017-02-02,2
100458,28.425833,-81.395,2017-02-03,2
100459,28.425833,-81.395,2017-02-04,2
100460,28.425833,-81.395,2017-02-05,2
100461,28.425833,-81.395,2017-02-06,2


In [9]:
# Step 4: assign final window_ids after filtering
window_meta = (
    df_non_overlap.groupby(["latitude", "longitude", "window_in_coord"], as_index=False)
    .size()
    .rename(columns={"size": "n_rows"})
    .sort_values(["latitude", "longitude", "window_in_coord"])
    .reset_index(drop=True)
)

window_meta["window_number"] = window_meta.groupby(["latitude", "longitude"]).cumcount()
window_meta["window_id"] = (
    window_meta["latitude"].astype(str) + "_" +
    window_meta["longitude"].astype(str) + "_" +
    window_meta["window_number"].astype(str)
)

df_final = df_non_overlap.merge(
    window_meta[["latitude", "longitude", "window_in_coord", "window_id"]],
    on=["latitude", "longitude", "window_in_coord"],
    how="left",
)

df_final = df_final.sort_values(["window_id", "source_row"]).reset_index(drop=True)

check_sizes = df_final.groupby("window_id").size()
assert (check_sizes == SEQ_LEN).all(), "Some final windows are not exactly 75 rows."
assert not df_final.duplicated(["latitude", "longitude", "datetime"]).any(), "Overlap still exists."

print(f"Final windows: {df_final['window_id'].nunique():,}")
print(f"Final rows: {len(df_final):,}")
display(df_final[["latitude", "longitude", "datetime", "Wildfire", "window_id", "source_row"]].head(10))

Final windows: 125,995
Final rows: 9,449,625


,latitude,longitude,datetime,Wildfire,window_id,source_row
0,25.26027,-80.80099,2016-04-10,0,25.2602699992712_-80.80099_0,297300
1,25.26027,-80.80099,2016-04-11,0,25.2602699992712_-80.80099_0,297301
2,25.26027,-80.80099,2016-04-12,0,25.2602699992712_-80.80099_0,297302
3,25.26027,-80.80099,2016-04-13,0,25.2602699992712_-80.80099_0,297303
4,25.26027,-80.80099,2016-04-14,0,25.2602699992712_-80.80099_0,297304
5,25.26027,-80.80099,2016-04-15,0,25.2602699992712_-80.80099_0,297305
6,25.26027,-80.80099,2016-04-16,0,25.2602699992712_-80.80099_0,297306
7,25.26027,-80.80099,2016-04-17,0,25.2602699992712_-80.80099_0,297307
8,25.26027,-80.80099,2016-04-18,0,25.2602699992712_-80.80099_0,297308
9,25.26027,-80.80099,2016-04-19,0,25.2602699992712_-80.80099_0,297309


## Validate Ignition-Day Labels

Checks:
- days 1-60 must all be `Wildfire = 0`
- if day 61 is `0`, then days 61-75 must all be `0`
- if day 61 is `1`, then days 61-75 must all be `1`

In [10]:
def validate_window_labels(g: pd.DataFrame) -> dict:
    g = g.sort_values("source_row").reset_index(drop=True)
    labels = g["Wildfire"].astype(int).to_numpy()

    first_60_all_no = bool((labels[:60] == 0).all())
    day_61_label = int(labels[60])
    last_15_all_no = bool((labels[60:] == 0).all())
    last_15_all_yes = bool((labels[60:] == 1).all())

    if day_61_label == 0:
        case_type = "no_fire"
        pattern_ok = first_60_all_no and last_15_all_no
    else:
        case_type = "fire"
        pattern_ok = first_60_all_no and last_15_all_yes

    return {
        "window_id": g["window_id"].iloc[0],
        "case_type": case_type,
        "day_61_label": day_61_label,
        "first_60_all_no": first_60_all_no,
        "last_15_all_no": last_15_all_no,
        "last_15_all_yes": last_15_all_yes,
        "pattern_ok": pattern_ok,
    }

validation_df = pd.DataFrame(
    [validate_window_labels(g) for _, g in df_final.groupby("window_id", sort=False)]
)

print("Case counts:")
print(validation_df["case_type"].value_counts().to_string())
print()
print("Validation results:")
print(validation_df["pattern_ok"].value_counts().to_string())

bad_windows = validation_df.loc[~validation_df["pattern_ok"]].copy()
print(f"\nInvalid windows: {len(bad_windows):,}")
display(bad_windows.head(20))

Case counts:
case_type
no_fire    93252
fire       32743

Validation results:
pattern_ok
True    125995

Invalid windows: 0


,window_id,case_type,day_61_label,first_60_all_no,last_15_all_no,last_15_all_yes,pattern_ok


In [11]:
# Inspect the first invalid window, if any
if bad_windows.empty:
    print("All final windows follow the expected ignition-day label rule.")
else:
    bad_window_id = bad_windows.iloc[0]["window_id"]
    display(
        df_final.loc[df_final["window_id"] == bad_window_id, ["window_id", "datetime", "Wildfire", "source_row"]]
        .sort_values("source_row")
        .reset_index(drop=True)
    )

All final windows follow the expected ignition-day label rule.


## Reduce Windows to 61 Days

Each validated window is truncated to its first 61 rows so the modeling setup is:
- days 1-60: input features
- day 61: prediction label

In [14]:
WINDOW_LEN_FINAL = 61

df_model = (
    df_final.sort_values(["window_id", "source_row"])
    .groupby("window_id", group_keys=False, sort=False)
    .head(WINDOW_LEN_FINAL)
    .reset_index(drop=True)
)

window_sizes_final = df_model.groupby("window_id").size()
assert (window_sizes_final == WINDOW_LEN_FINAL).all(), "Some final windows are not exactly 61 rows."

print(f"Final 61-day windows: {df_model['window_id'].nunique():,}")
print(f"Final 61-day rows: {len(df_model):,}")
display(df_model[["window_id", "datetime", "Wildfire", "source_row"]].head(10))

Final 61-day windows: 125,995
Final 61-day rows: 7,685,695


,window_id,datetime,Wildfire,source_row
0,25.2602699992712_-80.80099_0,2016-04-10,0,297300
1,25.2602699992712_-80.80099_0,2016-04-11,0,297301
2,25.2602699992712_-80.80099_0,2016-04-12,0,297302
3,25.2602699992712_-80.80099_0,2016-04-13,0,297303
4,25.2602699992712_-80.80099_0,2016-04-14,0,297304
5,25.2602699992712_-80.80099_0,2016-04-15,0,297305
6,25.2602699992712_-80.80099_0,2016-04-16,0,297306
7,25.2602699992712_-80.80099_0,2016-04-17,0,297307
8,25.2602699992712_-80.80099_0,2016-04-18,0,297308
9,25.2602699992712_-80.80099_0,2016-04-19,0,297309


## Check Fill Values

Audit the final 61-day windows for any remaining `32767.0` sentinel values in the feature columns.

In [15]:
FILL_VALUE = 32767.0

fill_mask = df_model[FEATURES].eq(FILL_VALUE)
rows_with_fill = df_model.loc[fill_mask.any(axis=1)].copy()
windows_with_fill = rows_with_fill["window_id"].nunique()

print(f"Rows with fill value {FILL_VALUE}: {len(rows_with_fill):,}")
print(f"Windows with fill value {FILL_VALUE}: {windows_with_fill:,}")

if rows_with_fill.empty:
    print("No final 61-day window contains the fill value.")
else:
    feature_fill_counts = fill_mask.sum().sort_values(ascending=False)
    print("\nFill-value counts by feature:")
    print(feature_fill_counts[feature_fill_counts > 0].to_string())

    rows_with_fill["fill_features"] = fill_mask.loc[rows_with_fill.index].apply(
        lambda row: [col for col, has_fill in row.items() if has_fill],
        axis=1,
    )

    display(
        rows_with_fill[["window_id", "datetime", "source_row", "fill_features"] + FEATURES]
        .head(20)
    )

Rows with fill value 32767.0: 20,923
Windows with fill value 32767.0: 343

Fill-value counts by feature:
pr        20923
rmax      20923
rmin      20923
sph       20923
srad      20923
tmmn      20923
tmmx      20923
vs        20923
bi        20923
fm100     20923
fm1000    20923
erc       20923
etr       20923
pet       20923
vpd       20923


,window_id,datetime,source_row,fill_features,pr,rmax,rmin,sph,srad,tmmn,tmmx,vs,bi,fm100,fm1000,erc,etr,pet,vpd
5103321,41.647242586838985_-82.97884516695564_0,2019-08-04,7680825,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0
5103322,41.647242586838985_-82.97884516695564_0,2019-08-05,7680826,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0
5103323,41.647242586838985_-82.97884516695564_0,2019-08-06,7680827,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0
5103324,41.647242586838985_-82.97884516695564_0,2019-08-07,7680828,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0
5103325,41.647242586838985_-82.97884516695564_0,2019-08-08,7680829,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0
5103326,41.647242586838985_-82.97884516695564_0,2019-08-09,7680830,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0
5103327,41.647242586838985_-82.97884516695564_0,2019-08-10,7680831,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0
5103328,41.647242586838985_-82.97884516695564_0,2019-08-11,7680832,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0
5103329,41.647242586838985_-82.97884516695564_0,2019-08-12,7680833,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0
5103330,41.647242586838985_-82.97884516695564_0,2019-08-13,7680834,"[pr, rmax, rmin, sph, srad, tmmn, tmmx, vs, bi...",32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0,32767.0


## Leakage-Safe Input Imputation

This imputes only days 1-60 in each 61-row window.
- `32767.0` is converted to `NaN`
- interpolation is done within each window over time
- edge gaps use forward/backward fill
- any remaining gaps use per-feature medians computed from input rows only

Note: because this notebook does not split train/holdout yet, the fallback medians below are computed from all day 1-60 rows in `df_model`. Once you split the data, recompute these medians on the training split only.

In [18]:
df_model_clean = df_model.copy()
df_model_clean[FEATURES] = df_model_clean[FEATURES].replace(FILL_VALUE, np.nan)

# Build fallback medians from input rows only (days 1-60).
input_rows = (
    df_model_clean.sort_values(["window_id", "source_row"])
    .groupby("window_id", group_keys=False, sort=False)
    .head(60)
)
input_feature_medians = input_rows[FEATURES].median()
print("Fallback medians computed from day 1-60 rows.")
display(input_feature_medians.to_frame("median").T)

Fallback medians computed from day 1-60 rows.


,pr,rmax,rmin,sph,srad,tmmn,tmmx,vs,bi,fm100,fm1000,erc,etr,pet,vpd
median,0.0,80.0,30.9,0.00548,228.3,281.0,295.4,3.4,33.0,12.9,14.9,38.0,5.4,4.0,0.91


In [19]:
def impute_window_inputs(g: pd.DataFrame, feature_cols, medians: pd.Series) -> pd.DataFrame:
    g = g.sort_values("source_row").reset_index(drop=True).copy()

    # Only impute the first 60 rows used as model inputs.
    x = g.loc[:59, feature_cols].copy()
    x = x.interpolate(limit_direction="both")
    x = x.ffill().bfill()
    x = x.fillna(medians)

    g.loc[:59, feature_cols] = x
    return g

df_model_imputed = pd.concat(
    [
        impute_window_inputs(g, FEATURES, input_feature_medians)
        for _, g in df_model_clean.groupby("window_id", sort=False)
    ],
    axis=0,
    ignore_index=True,
)

df_model_imputed = df_model_imputed.sort_values(["window_id", "source_row"]).reset_index(drop=True)
display(df_model_imputed[["window_id", "datetime", "Wildfire", "source_row"] + FEATURES].head(5))

,window_id,datetime,Wildfire,source_row,pr,rmax,rmin,sph,srad,tmmn,tmmx,vs,bi,fm100,fm1000,erc,etr,pet,vpd
0,25.2602699992712_-80.80099_0,2016-04-10,0,297300,0.0,91.2,44.7,0.01111,306.6,290.7,300.4,7.3,37.0,16.4,18.6,27.0,8.0,5.9,0.71
1,25.2602699992712_-80.80099_0,2016-04-11,0,297301,0.0,100.0,51.0,0.01295,242.3,290.1,301.0,5.9,32.0,16.7,18.5,25.0,5.9,4.6,0.75
2,25.2602699992712_-80.80099_0,2016-04-12,0,297302,0.0,100.0,45.2,0.01282,299.7,289.6,302.7,4.0,27.0,16.7,18.4,27.0,6.6,5.3,0.79
3,25.2602699992712_-80.80099_0,2016-04-13,0,297303,0.0,100.0,47.0,0.01364,326.5,290.0,303.1,3.4,26.0,16.8,18.3,27.0,6.5,5.5,0.74
4,25.2602699992712_-80.80099_0,2016-04-14,0,297304,1.0,100.0,49.7,0.01437,292.7,292.0,303.2,3.8,27.0,16.9,18.2,27.0,6.4,5.2,0.78


In [20]:
# Validation after imputation
window_sizes_after_impute = df_model_imputed.groupby("window_id").size()
assert (window_sizes_after_impute == WINDOW_LEN_FINAL).all(), "Some windows changed size after imputation."

remaining_fill_input = (
    df_model_imputed.sort_values(["window_id", "source_row"])
    .groupby("window_id", group_keys=False, sort=False)
    .head(60)[FEATURES]
    .isna()
    .any()
    .any()
)
assert not remaining_fill_input, "NaN values still exist in the first 60 rows after imputation."

remaining_sentinel = df_model_imputed[FEATURES].eq(FILL_VALUE).any().any()
assert not remaining_sentinel, "Sentinel fill values still exist after cleaning."

print("Imputation complete.")
print(f"Final windows after imputation: {df_model_imputed['window_id'].nunique():,}")
print(f"Final rows after imputation: {len(df_model_imputed):,}")
print("No sentinel values remain, and all input rows (days 1-60) are fully imputed.")

Imputation complete.
Final windows after imputation: 125,995
Final rows after imputation: 7,685,695
No sentinel values remain, and all input rows (days 1-60) are fully imputed.


## Export Parquet

Export the final 61-day imputed dataset to Parquet.

In [21]:
OUT_PARQUET = "Wildfire_Dataset_61Day_Imputed.parquet"

try:
    import pyarrow  # noqa: F401
except ImportError:
    try:
        import fastparquet  # noqa: F401
    except ImportError as exc:
        raise ImportError(
            "Parquet export requires pyarrow or fastparquet in the notebook kernel."
        ) from exc

df_model_imputed.to_parquet(OUT_PARQUET, index=False)
print(f"Saved {OUT_PARQUET}")
print(f"Rows: {len(df_model_imputed):,}")
print(f"Windows: {df_model_imputed['window_id'].nunique():,}")

Saved Wildfire_Dataset_61Day_Imputed.parquet
Rows: 7,685,695
Windows: 125,995


## Final Dataset Statistics

Report summary statistics for the final 61-day imputed dataset, including the day-61 label distribution and feature summaries.

In [22]:
final_window_sizes = df_model_imputed.groupby("window_id").size()
day_61_df = (
    df_model_imputed.sort_values(["window_id", "source_row"])
    .groupby("window_id", group_keys=False, sort=False)
    .nth(60)
    .reset_index()
)
input_days_df = (
    df_model_imputed.sort_values(["window_id", "source_row"])
    .groupby("window_id", group_keys=False, sort=False)
    .head(60)
    .reset_index(drop=True)
)

print("Dataset shape:")
print(f"Rows: {len(df_model_imputed):,}")
print(f"Windows: {df_model_imputed['window_id'].nunique():,}")
print(f"Window size min/max: {final_window_sizes.min()} / {final_window_sizes.max()}")
print()
print("Day-61 label distribution:")
label_counts = day_61_df["Wildfire"].value_counts().sort_index()
label_rates = day_61_df["Wildfire"].value_counts(normalize=True).sort_index()
label_summary = pd.DataFrame({
    "count": label_counts,
    "rate": label_rates,
})
display(label_summary)

print("Input-day feature summary (days 1-60):")
display(input_days_df[FEATURES].describe().T)

print("Day-61 feature summary:")
display(day_61_df[FEATURES].describe().T)

Dataset shape:
Rows: 7,685,695
Windows: 125,995
Window size min/max: 61 / 61

Day-61 label distribution:


,count,rate
Wildfire,,
0,93252,0.740125
1,32743,0.259875


Input-day feature summary (days 1-60):


,count,mean,std,min,25%,50%,75%,max
pr,7559700.0,1.711320,6.261086,0.00000,0.00000,0.00000,0.00000,528.00000
rmax,7559700.0,76.261574,20.727641,5.00000,62.00000,80.00000,95.10000,100.00000
rmin,7559700.0,33.649323,18.322885,1.00000,19.50000,30.90000,45.50000,100.00000
sph,7559700.0,0.006051,0.003377,0.00013,0.00354,0.00548,0.00775,0.02382
srad,7559700.0,221.800899,88.251071,0.00000,147.40000,228.30000,300.80000,440.60000
tmmn,7559700.0,280.184740,8.887770,230.90000,274.60000,281.00000,286.30000,310.00000
tmmx,7559700.0,294.141010,10.358275,241.90000,287.70000,295.40000,301.80000,324.30000
vs,7559700.0,3.767754,1.675756,0.30000,2.60000,3.40000,4.60000,19.60000
bi,7559700.0,33.473250,23.837298,0.00000,19.00000,33.00000,49.00000,189.00000
fm100,7559700.0,12.826002,4.946016,1.10000,8.70000,12.90000,16.60000,32.60000


Day-61 feature summary:


,count,mean,std,min,25%,50%,75%,max
pr,125652.0,1.690497,6.155503,0.00000,0.00000,0.00000,0.00000,265.70000
rmax,125652.0,75.212929,20.888377,8.70000,60.50000,78.50000,93.90000,100.00000
rmin,125652.0,32.237046,17.989014,1.00000,18.50000,29.10000,43.50000,100.00000
sph,125652.0,0.006082,0.003202,0.00026,0.00372,0.00556,0.00773,0.02341
srad,125652.0,221.502851,86.299490,0.00000,149.30000,227.10000,296.90000,410.10000
tmmn,125652.0,280.818139,8.115272,236.40000,275.50000,281.40000,286.40000,306.40000
tmmx,125652.0,295.037168,9.623135,245.50000,288.90000,295.90000,302.10000,321.70000
vs,125652.0,3.773770,1.684680,0.30000,2.60000,3.40000,4.60000,17.40000
bi,125652.0,34.568944,24.087240,0.00000,20.00000,35.00000,50.00000,163.00000
fm100,125652.0,12.491762,4.861150,1.60000,8.50000,12.50000,16.10000,32.10000


In [23]:
print("Missing values after imputation:")
missing_summary = pd.DataFrame({
    "input_days_missing": input_days_df[FEATURES].isna().sum(),
    "day_61_missing": day_61_df[FEATURES].isna().sum(),
}).sort_index()
display(missing_summary)

print("Feature quantiles for input days (days 1-60):")
display(input_days_df[FEATURES].quantile([0.01, 0.05, 0.5, 0.95, 0.99]).T)

print("Feature quantiles for day 61:")
display(day_61_df[FEATURES].quantile([0.01, 0.05, 0.5, 0.95, 0.99]).T)

Missing values after imputation:


,input_days_missing,day_61_missing
bi,0,343
erc,0,343
etr,0,343
fm100,0,343
fm1000,0,343
pet,0,343
pr,0,343
rmax,0,343
rmin,0,343
sph,0,343


Feature quantiles for input days (days 1-60):


,0.01,0.05,0.50,0.95,0.99
pr,0.000,0.00000,0.00000,10.00000,29.90000
rmax,25.200,36.80000,80.00000,100.00000,100.00000
rmin,4.100,8.60000,30.90000,67.60000,83.20000
sph,0.001,0.00183,0.00548,0.01279,0.01704
srad,42.400,73.70000,228.30000,347.40000,362.70000
tmmn,254.400,264.90000,281.00000,293.70000,297.30000
tmmx,266.300,275.20000,295.40000,308.80000,313.00000
vs,1.200,1.70000,3.40000,7.00000,9.10000
bi,0.000,0.00000,33.00000,74.00000,93.00000
fm100,3.700,5.10000,12.90000,20.80000,23.80000


Feature quantiles for day 61:


,0.01,0.05,0.50,0.95,0.99
pr,0.00000,0.00000,0.00000,10.10000,29.50000
rmax,24.30000,36.20000,78.50000,100.00000,100.00000
rmin,3.70000,8.20000,29.10000,66.20000,81.90000
sph,0.00124,0.00204,0.00556,0.01227,0.01665
srad,44.20000,76.30000,227.10000,347.30000,362.10000
tmmn,259.20000,267.30000,281.40000,293.50000,297.30000
tmmx,270.15100,277.70000,295.90000,309.10000,313.20000
vs,1.20000,1.70000,3.40000,7.00000,9.10000
bi,0.00000,0.00000,35.00000,75.00000,94.00000
fm100,3.60000,5.00000,12.50000,20.40000,23.60000
